# 04 Entry

Build the firm x product entry panel for the prediction step.

Design (no look-ahead):
- A firm-product link exists from the year of its first PQR transaction.
- For each cutoff year T, the "state" is every link that started before T. Proximity (phi)
  between products is computed only from that state.
- Risk set at T: firms active before T x products bought before T, where the firm does not
  yet supply the product.
- Outcome: entry = the firm's first transaction for that product falls in [T, T + H).
- Main predictor: density_fp = sum_{j in firm's products} phi_pj / sum_j phi_pj  (Hidalgo et al. 2007),
  the share of product p's proximity mass that the firm already covers.
Output: data/processed/entry_panel.csv


Entry models: does network proximity predict which products a firm enters next?

1. Explanatory logits on the pooled panel (cutoffs 2011-2023), SE clustered by firm.
2. Out-of-sample prediction: train on cutoffs 2011-2017, test on 2020 and 2023.
   Compare a popularity baseline, a controls-only logit, logit + density, and a random forest.
Sample: firms with a PQR transaction in the 3 years before the cutoff x products bought in
those 3 years, excluding pairs the firm already supplies.


Score potential new suppliers for thinly supplied products (policy application).

- Fit the full entry logit (density + same area + firm breadth + product popularity + cutoff FE)
  on all cutoffs 2011-2023.
- Build the current state (all links through 2025, cutoff 2026) and score every
  recently active firm x recently bought product the firm does not yet supply.
- Thin products: bought in 2021-2025 with <= 2 manufacturers supplying them in 2021-2025.
- Flag whether the candidate already holds a WHO PQ listing whose INN text contains every
  molecule in the product name (a rough signal of regulatory readiness).
Output: outputs/tables/t12_candidate_suppliers.csv


f8: entry rate by network density; f9: out-of-sample recall by model.

In [ ]:
%run 00_utils.ipynb

import re
import numpy as np, pandas as pd, statsmodels.formula.api as smf, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

set_style()

## Panel

In [ ]:
HORIZON = 3
CUTOFFS = [2011, 2014, 2017, 2020, 2023]
pqr = load_clean("pqr_medicines")
pqr = pqr[pqr.year.between(2005, 2025)]
links = pqr.groupby(["firm", "product"]).agg(first_year=("year", "min"), n_tx=("pk", "size")).reset_index()
product_area = pqr.groupby("product")["area"].first()
firm_hq = pqr.groupby("firm")["hq_country"].first()

# build all firm-product pairs at risk of entry at one cutoff
def panel_at(cutoff_year):
    # only links from before the cutoff, so nothing looks ahead
    state = links[links.first_year < cutoff_year]
    proximity_edges = project_products(state)
    phi = pd.concat([proximity_edges.rename(columns={"source": "p", "target": "q"}),
                     proximity_edges.rename(columns={"target": "p", "source": "q"})])[["p", "q", "phi"]]
    total_proximity = phi.groupby("p")["phi"].sum()
    active_firms, known_products = state.firm.unique(), state["product"].unique()
    existing_links = set(zip(state.firm, state["product"]))
    grid = pd.MultiIndex.from_product([active_firms, known_products], names=["firm", "product"]).to_frame(index=False)
    grid = grid[[(f, p) not in existing_links for f, p in zip(grid.firm, grid["product"])]]
    # density: proximity from candidate product p to products the firm already supplies
    firm_products = state[["firm", "product"]].rename(columns={"product": "q"})
    covered = firm_products.merge(phi, on="q").groupby(["firm", "p"])["phi"].sum().rename("covered").reset_index()
    pairs = grid.merge(covered, left_on=["firm", "product"], right_on=["firm", "p"], how="left").drop(columns="p")
    pairs["covered"] = pairs["covered"].fillna(0)
    pairs["density"] = pairs["covered"] / pairs["product"].map(total_proximity).replace(0, np.nan)
    pairs["density"] = pairs["density"].fillna(0)
    # controls
    pairs["firm_n_products"] = pairs.firm.map(state.groupby("firm")["product"].nunique())
    pairs["prod_n_suppliers"] = pairs["product"].map(state.groupby("product")["firm"].nunique())
    pairs["area"] = pairs["product"].map(product_area)
    firm_areas = state.assign(area=state["product"].map(product_area)).groupby("firm")["area"].agg(set)
    pairs["same_area"] = [risk_set in firm_areas[f] for f, risk_set in zip(pairs.firm, pairs.area)]
    recent_pqr = pqr[pqr.year.between(cutoff_year - 3, cutoff_year - 1)]
    pairs["firm_active_recent"] = pairs.firm.isin(recent_pqr.firm)
    pairs["prod_bought_recent"] = pairs["product"].isin(recent_pqr["product"])
    pairs["hq_country"] = pairs.firm.map(firm_hq)
    # outcome: the firm starts supplying the product within the horizon
    future_links = links[(links.first_year >= cutoff_year) & (links.first_year < cutoff_year + HORIZON)]
    pairs["entry"] = [(f, p) in set(zip(future_links.firm, future_links["product"])) for f, p in zip(pairs.firm, pairs["product"])]
    pairs["cutoff"] = cutoff_year
    return pairs

panel = pd.concat([panel_at(cutoff_year) for cutoff_year in CUTOFFS], ignore_index=True)
panel.to_csv(PROCESSED / "entry_panel.csv", index=False)
cutoff_summary = panel.groupby("cutoff").agg(rows=("entry", "size"), entries=("entry", "sum"),
                                firms=("firm", "nunique"), products=("product", "nunique"),
                                mean_density=("density", "mean"))
cutoff_summary["entry_rate_%"] = (100 * cutoff_summary.entries / cutoff_summary.rows).round(2)
print(cutoff_summary.round(3).to_string())
risk_set = panel[panel.firm_active_recent & panel.prod_bought_recent]
print("\nrestricted to recently active firms & recently bought products:", len(risk_set), "rows,", risk_set.entry.sum(), "entries")
print("density: entrants vs non-entrants (median):", risk_set.groupby("entry").density.median().round(3).to_dict())
print("entries in same disease area:", round(risk_set[risk_set.entry].same_area.mean(), 2), "| base rate same_area:", round(risk_set.same_area.mean(), 2))

I used 5 cutoffs and only built proximity from links that started before each cutoff, so the predictor did not use future information. After I limited the sample to recently active firms and recently bought products, the panel had 24,111 rows and 142 entries (0.6%).

Entrants had much higher density than non-entrants (median 0.27 vs 0.01). 92% of entries were in a disease area the firm already served, compared with a base rate of 44%. Products first bought after a cutoff are outside the risk set, so entry into brand-new products is not captured.

## Models

In [ ]:
def load_risk_set():
    """Read the panel, keep recently active pairs, and add model variables."""
    panel = pd.read_csv(PROCESSED / "entry_panel.csv")
    return prepare_pairs(panel[panel.firm_active_recent & panel.prod_bought_recent]).assign(entry=lambda df: df.entry.astype(int))


def prepare_pairs(pairs):
    """Add the numeric variables used by the logit models."""
    return pairs.assign(same_area=pairs.same_area.astype(int), log_firm_products=np.log(pairs.firm_n_products),
                        log_prod_suppliers=np.log(pairs.prod_n_suppliers))


risk_set = load_risk_set()
risk_set["density_z"] = (risk_set.density - risk_set.density.mean()) / risk_set.density.std()
risk_set["cutoff_f"] = risk_set.cutoff.astype(str)

In [ ]:
# ---- 1. explanatory logits
formulas = {"M1 controls": "entry ~ log_firm_products + log_prod_suppliers + C(cutoff_f)",
         "M2 + same area": "entry ~ same_area + log_firm_products + log_prod_suppliers + C(cutoff_f)",
         "M3 + density": "entry ~ density_z + log_firm_products + log_prod_suppliers + C(cutoff_f)",
         "M4 full": "entry ~ density_z + same_area + log_firm_products + log_prod_suppliers + C(cutoff_f)"}
logit_rows = []
for name, formula in formulas.items():
    logit_fit = smf.logit(formula, data=risk_set).fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(risk_set.firm)[0]})
    for term in ["density_z", "same_area", "log_firm_products", "log_prod_suppliers"]:
        if term in logit_fit.params:
            logit_rows.append({"model": name, "term": term, "odds_ratio": np.exp(logit_fit.params[term]),
                         "ci_low": np.exp(logit_fit.conf_int().loc[term, 0]), "ci_high": np.exp(logit_fit.conf_int().loc[term, 1]),
                         "p": logit_fit.pvalues[term]})
    logit_rows.append({"model": name, "term": "pseudo_R2", "odds_ratio": logit_fit.prsquared})
    logit_rows.append({"model": name, "term": "N / entries", "odds_ratio": len(risk_set), "ci_low": risk_set.entry.sum()})
logit_table = pd.DataFrame(logit_rows)
logit_table.round(4).to_csv(TABLES / "t10_entry_logit.csv", index=False)
print(logit_table.round(3).to_string(index=False))

Density mattered beyond disease area. In the full model, one SD of density raised the odds of entry by 37% (OR 1.37, 95% CI 1.24-1.52), holding constant same area (OR 8.1), firm breadth (OR 2.1) and product popularity (OR 2.2). Pseudo R2 rose from 0.22 with same area and controls to 0.24 with density. Same area had the larger effect, and density added a smaller but clear signal.

In [ ]:
# ---- 2. out-of-sample
control_features = ["log_firm_products", "log_prod_suppliers", "same_area"]
full_features = control_features + ["density"]
# train on earlier cutoffs and test on later ones
train, test = risk_set[risk_set.cutoff <= 2017], risk_set[risk_set.cutoff >= 2020]
def top_share_recall(outcomes, scores, share):
    n_top = max(1, int(round(share * len(outcomes))))
    top_rows = np.argsort(-scores)[:n_top]
    return outcomes[top_rows].sum() / outcomes.sum()                       # recall of entries within top-k%
model_results = []
candidate_models = {
    "Popularity (n suppliers)": (["log_prod_suppliers"], LogisticRegression(max_iter=1000)),
    "Controls logit": (control_features, LogisticRegression(max_iter=1000)),
    "Density logit": (full_features, LogisticRegression(max_iter=1000)),
    "Random forest": (full_features, RandomForestClassifier(n_estimators=500, min_samples_leaf=20,
                                                     class_weight="balanced", random_state=5205, n_jobs=-1)),
}
for name, (features, estimator) in candidate_models.items():
    estimator.fit(train[features], train.entry)
    scores = estimator.predict_proba(test[features])[:, 1]
    outcomes = test.entry.to_numpy()
    model_results.append({"model": name, "AUC": roc_auc_score(outcomes, scores), "avg_precision": average_precision_score(outcomes, scores),
                "recall_top1%": top_share_recall(outcomes, scores, .01), "recall_top5%": top_share_recall(outcomes, scores, .05), "recall_top10%": top_share_recall(outcomes, scores, .10)})
model_results = pd.DataFrame(model_results)
model_results["base_rate"] = test.entry.mean()
model_results.round(3).to_csv(TABLES / "t11_entry_prediction.csv", index=False)
print(f"\ntrain {len(train)} rows / {train.entry.sum()} entries | test {len(test)} rows / {test.entry.sum()} entries")
print(model_results.round(3).to_string(index=False))

# entry rate by density decile (for figure)
risk_set["density_bin"] = pd.cut(risk_set.density, [-0.001, 0, .05, .1, .2, .3, .5, 1.0],
                          labels=["0", "0-0.05", "0.05-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.5", "0.5+"])
density_rates = risk_set.groupby("density_bin", observed=True).agg(pairs=("entry", "size"), entries=("entry", "sum"))
density_rates["entry_rate_%"] = 100 * density_rates.entries / density_rates.pairs
density_rates.round(2).to_csv(TABLES / "t11b_entry_by_density.csv")
print("\n", density_rates.round(2).to_string())

I trained on the 2011-2017 cutoffs and tested on 2020 and 2023, which had 34 entries (base rate 0.5%). Popularity alone was close to random (AUC 0.55). The controls logit reached AUC 0.87, and adding density raised it to 0.89. Density also found more entries among the top predictions: 47% vs 35% in the top 5%, and 62% vs 56% in the top 10%. The random forest had a lower AUC (0.83) and lower top-5% and top-10% recall than the density logit, so I kept the simpler logit. The gain from density is modest, and 34 test entries is a small sample.

Entry rose with density across the bins, from 0.15% at zero density to 7.0% at 0.3-0.5 and 8.1% above 0.5.

## Candidate suppliers

In [ ]:
risk_set = load_risk_set()
# fit the full model on all historical cutoffs
entry_logit = smf.logit("entry ~ density + same_area + log_firm_products + log_prod_suppliers", data=risk_set).fit(disp=0)

# score the current state as if 2026 were the next cutoff
current_pairs = prepare_pairs(panel_at(2026))
current_pairs = current_pairs[current_pairs.firm_active_recent & current_pairs.prod_bought_recent].copy()
current_pairs["p_entry_3y"] = entry_logit.predict(current_pairs)

all_pqr = load_clean("pqr_medicines")
recent_pqr = all_pqr[all_pqr.year.between(2021, 2025)]
current_supply = recent_pqr.groupby("product").agg(current_suppliers=("firm", lambda s: "; ".join(sorted(s.unique()))),
                                 n_current=("firm", "nunique"), spend_2021_25_usd_m=("cost_usd", lambda x: x.sum() / 1e6))
# thin products have at most two recent suppliers
thin_products = current_supply[current_supply.n_current <= 2]

pq_products = load_clean("pq_fpp")
pq_text = pq_products.groupby("firm")["INN, Dosage Form and Strength"].apply(lambda s: " | ".join(s).lower())
# rough WHO PQ check: every molecule in the product name appears in the firm's listings
NON_MOLECULE_WORDS = {"tablet", "tablets", "coblister", "blister", "patient", "dispersible", "pediatric", "paediatric",
        "india", "category", "granules", "injection", "sulfate", "phosphate", "tetraphosphate"}
def molecules(product_name):
    tokens = re.findall(r"[a-z]{5,}", re.sub(r"\(.*?\)", " ", product_name.lower()))
    return [token for token in tokens if token not in NON_MOLECULE_WORDS]
def pq_ready(firm_name, product_name):
    needed_molecules = molecules(product_name)
    listed_text = pq_text.get(firm_name, "")
    return bool(needed_molecules) and all(token in listed_text for token in needed_molecules)

# keep the five most likely entrants for each thin product
candidates = current_pairs[current_pairs["product"].isin(thin_products.index)].copy()
candidates["who_pq_listed"] = [pq_ready(firm_id, product_id) for firm_id, product_id in zip(candidates.firm, candidates["product"])]
candidates = candidates.sort_values(["product", "p_entry_3y"], ascending=[True, False]).groupby("product").head(5)
candidates = candidates.merge(thin_products, left_on="product", right_index=True)
candidates["rank"] = candidates.groupby("product").cumcount() + 1
candidate_table = candidates[["area", "product", "n_current", "current_suppliers", "spend_2021_25_usd_m", "rank", "firm", "hq_country",
         "p_entry_3y", "density", "same_area", "who_pq_listed"]].sort_values(["spend_2021_25_usd_m", "rank"], ascending=[False, True])
candidate_table.round(4).to_csv(TABLES / "t12_candidate_suppliers.csv", index=False)

print("thin products (<=2 suppliers in 2021-25):", len(thin_products), "| with scored candidates:", candidate_table["product"].nunique())
pd.set_option("display.width", 250, "display.max_colwidth", 45)
preview = candidate_table[candidate_table["rank"] <= 3].groupby("product", sort=False).head(3)
top_products = candidate_table.drop_duplicates("product").head(8)["product"]
print(preview[preview["product"].isin(top_products)][["product", "current_suppliers", "spend_2021_25_usd_m", "rank", "firm",
                                            "p_entry_3y", "density", "who_pq_listed"]].round(3).to_string(index=False))
print("\ncandidates with WHO PQ listing for the molecule:", candidate_table.who_pq_listed.sum(), "of", len(candidate_table))

39 products bought in 2021-2025 had at most 2 manufacturers in that period. For 31 of them, I listed the 5 active firms with the highest predicted 3-year entry probability. Some examples:

- Bedaquiline (Janssen, Lupin): Macleods ranked first (p = 0.10) and has a WHO PQ listing
- Abacavir + dolutegravir + lamivudine (Aurobindo, Laurus): Cipla, Mylan and Macleods (p = 0.20-0.26), all with WHO PQ listings
- DHA-piperaquine (Alfasigma, Guilin): Cipla, Macleods and Ipca
- Pyronaridine-artesunate (Shin Poong only): density was 0 for every candidate, so the network gives no signal

Products like pyronaridine-artesunate are where market shaping could not rely on the network to point to likely entrants. The WHO PQ flag is a text match on molecule names and should be checked by hand.

## Figures

In [ ]:
density_rates = pd.read_csv(TABLES / "t11b_entry_by_density.csv")
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar(density_rates.density_bin, density_rates["entry_rate_%"], color=BLUE, width=0.72, edgecolor="white", linewidth=1)
for bin_position, row in density_rates.iterrows():
    ax.text(bin_position, row["entry_rate_%"] + 0.15, f"{row['entry_rate_%']:.1f}%", ha="center", fontsize=8.5, color=INK2)
    ax.text(bin_position, -0.75, f"n={int(row.pairs):,}", ha="center", fontsize=7.5, color=MUTED)
ax.set_ylim(-1.1, density_rates["entry_rate_%"].max() * 1.18)
ax.axhline(0, color=GRID, lw=0.8)
ax.set_xlabel("Density: how close the product is to what the firm already supplies", labelpad=14)
ax.set_ylabel("Firms entering within 3 years (%)")
ax.grid(axis="x", visible=False)
add_title(ax, "Firms enter products close to what they already make",
          "Share of firm-product pairs where the firm starts supplying the product in the next 3 years")
fig.text(0.01, -0.13, f"Pooled over cutoffs {', '.join(map(str, CUTOFFS))} ({len(risk_set):,} pairs, {risk_set.entry.sum()} entries). Density = share of the product's "
         "proximity links that the firm already covers.\nSource: Global Fund PQR.", fontsize=7.5, color=MUTED)
save_figure(fig, "f8_entry_by_density.png")

In [ ]:
prediction_results = pd.read_csv(TABLES / "t11_entry_prediction.csv")
fig, ax = plt.subplots(figsize=(7.5, 3.4))
model_names = prediction_results.model[::-1]
recall = prediction_results["recall_top10%"][::-1] * 100
bar_colors = [BLUE if model == "Density logit" else GRAY for model in model_names]
ax.barh(model_names, recall, color=bar_colors, height=0.6, edgecolor="white", linewidth=1)
for row, (recall_value, auc_value) in enumerate(zip(recall, prediction_results.AUC[::-1])):
    ax.text(recall_value + 1, row, f"{recall_value:.0f}%   (AUC {auc_value:.2f})", va="center", fontsize=8.5, color=INK2)
ax.set_xlim(0, 100)
ax.set_xlabel("Share of actual entries found in the top 10% of predictions (%)")
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
ax.set_title("Adding network density modestly improves prediction", loc="left", fontsize=12, fontweight="bold", pad=10)
fig.text(0.01, -0.08, f"Trained on cutoffs 2011-2017, tested on 2020 and 2023 ({len(test):,} pairs, {test.entry.sum()} entries; base rate {prediction_results.base_rate[0]:.1%}). "
         "Controls = firm breadth, product popularity, same disease area.", fontsize=7.5, color=MUTED)
save_figure(fig, "f9_entry_prediction.png")

There were only 142 entries, so the estimates are imprecise, and outcomes for the 2023 cutoff are cut short by reporting lags. PQR also only shows Global Fund-financed purchases, so a firm may have supplied a product elsewhere before it appears here as an entrant.